In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

In [2]:
def load_and_preprocess_data(file_path):
    # Load the dataset
    data = pd.read_csv('cleaned_data.csv')

    # Separate features (X) and target (y)
    X = data.drop(columns=['StudentID', 'Grade'])
    y = data['Grade']

    # Add polynomial features
    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_poly = poly.fit_transform(X)

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.2, random_state=42)

    # Standardize the numerical features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, scaler

In [3]:
def train_and_evaluate_random_forest(X_train, X_test, y_train, y_test):
    rf_param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    }
    rf_reg = RandomForestRegressor(random_state=42)
    rf_grid_search = GridSearchCV(estimator=rf_reg, param_grid=rf_param_grid, cv=5, scoring='neg_mean_squared_error', verbose=1)
    rf_grid_search.fit(X_train, y_train)
    best_rf_reg = rf_grid_search.best_estimator_
    y_pred = best_rf_reg.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return best_rf_reg, mse, mae, r2

In [6]:
def main():
    file_path = '/Users/Kadija/Desktop/Model/cleaned_data.csv'   
    X_train, X_test, y_train, y_test, scaler = load_and_preprocess_data(file_path)

    # Train and evaluate Random Forest Regressor
    best_rf_reg, rf_reg_mse, rf_reg_mae, rf_reg_r2 = train_and_evaluate_random_forest(X_train, X_test, y_train, y_test)
    print(f"Random Forest Regressor MSE: {rf_reg_mse:.2f}, MAE: {rf_reg_mae:.2f}, R2: {rf_reg_r2:.2f}")

 # Save the model and scaler
    joblib.dump(best_rf_reg, 'rf_reg_model.pkl')
    joblib.dump(scaler, 'scaler.pkl')

In [7]:
if __name__ == "__main__":
    main()

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Random Forest Regressor MSE: 2.25, MAE: 0.94, R2: 0.81
